In [ ]:
import os
import json
import random
import operator
from typing import Annotated, List, TypedDict, Literal, Dict
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, AIMessage

# ============================================================================
# 1. SETUP & CONFIGURATION
# ============================================================================

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic


OPENAI_API_KEY = ""  # Replace with your actual OpenAI API Key
ANTHROPIC_API_KEY = "" # Replace with your actual Anthropic API Key
# TOGGLE PROVIDER
LLM_PROVIDER = "anthropic"

if LLM_PROVIDER == "openai":
    llm = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY, temperature=0.7)
elif LLM_PROVIDER == "anthropic":
    llm = ChatAnthropic(model="claude-3-opus-20240229", api_key=ANTHROPIC_API_KEY, temperature=0.7)

# The 37 Dimensions
DIMENSIONS = [
    "maintaining_stable_weight", "managing_mood", "taking_medication",
    "participating_healthcare", "organizing_housework", "talking_to_people",
    "expressing_feelings", "managing_safety", "managing_risk",
    "sleep_schedule", "eating_schedule", "managing_work_school",
    "work_life_balance", "showing_up_appointments", "managing_finance",
    "adequate_nutrition", "problem_solving", "family_support",
    "family_relationship", "alcohol_use", "tobacco_use",
    "substance_use", "leisure_activities", "creativity",
    "community_participation", "social_support", "friend_relationships",
    "relationship_boundaries", "sexual_safety", "work_productivity",
    "work_motivation", "coping_skills", "self_harm_control",
    "law_abiding", "legal_issues", "personal_hygiene",
    "exercise_sports"
]

# ============================================================================
# Create patients.json with valid content
# ============================================================================

# Sample patient profiles (replace with your actual data if needed)
sample_patient_profiles = [
    {
        "id": "1",
        "name": "Alice",
        "condition": "Generalized Anxiety Disorder",
        "context": "Alice is a 28-year-old software engineer experiencing high stress at work. She often feels overwhelmed and struggles with decision-making.",
        "goals": "Reduce anxiety, improve coping mechanisms, manage work-life balance.",
        "boundaries": "Avoid discussing past trauma without explicit consent.",
        "risk_level": "medium"
    },
    {
        "id": "2",
        "name": "Bob",
        "condition": "Mild Depression",
        "context": "Bob is a 45-year-old teacher who recently went through a divorce. He feels isolated and lacks motivation for daily activities.",
        "goals": "Increase social interaction, regain interest in hobbies, improve mood.",
        "boundaries": "Do not pressure him to talk about the divorce details.",
        "risk_level": "high"
    },
    {
        "id": "3",
        "name": "Charlie",
        "condition": "Insomnia",
        "context": "Charlie is a 35-year-old artist with chronic sleep difficulties. He often feels fatigued and irritable.",
        "goals": "Establish a regular sleep schedule, improve sleep quality, manage daytime fatigue.",
        "boundaries": "Focus on current sleep habits, not childhood experiences.",
        "risk_level": "medium"
    },
    {
        "id": "4",
        "name": "Diana",
        "condition": "Healthy - Check-up",
        "context": "Diana is a 22-year-old student maintaining good mental health. She's interested in proactive wellness strategies.",
        "goals": "Maintain well-being, learn stress prevention techniques.",
        "boundaries": "Keep conversation light and positive.",
        "risk_level": "low"
    }
]

# Remove existing files to prevent conflicts
if os.path.exists("patients.json.rtf"):
    os.remove("patients.json.rtf")
if os.path.exists("patients.json"):
    os.remove("patients.json")

with open("patients.json", 'w') as f:
    json.dump(sample_patient_profiles, f, indent=2)


# ============================================================================
# 2. DATA LOADER
# ============================================================================

def load_patient_profiles(filename="patients.json"):
    """Loads patient profiles from JSON file."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: '{filename}' not found. Please create it first.")
        return []

# ============================================================================
# 3. Q-LEARNING AGENT
# ============================================================================

class QLearningAgent:
    def __init__(self, dimensions: List[str], epsilon=0.9, alpha=0.1, gamma=0.9):
        self.dimensions = dimensions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.q_table = {}
        self._initialize_q_table()

    def _initialize_q_table(self):
        # Initial heuristics - prioritizing generally high-risk areas
        high_priority = ["managing_mood", "sleep_schedule", "self_harm_control", "substance_use"]
        for state in ["START"] + self.dimensions:
            for action in self.dimensions:
                if action in high_priority:
                    self.q_table[(state, action)] = 0.8
                else:
                    self.q_table[(state, action)] = 0.5

    def select_next_dimension(self, current_dim: str, screened_dims: List[str]) -> str:
        available = [d for d in self.dimensions if d not in screened_dims]
        if not available: return None

        if random.random() < self.epsilon:
            q_values = {action: self.q_table.get((current_dim, action), 0.5) for action in available}
            return max(q_values, key=q_values.get)
        else:
            return random.choice(available)

    def update_q_value(self, prev_state: str, action: str, reward: int):
        # Score 2 = Reward 1.0 (High value information)
        r = 1.0 if reward == 2 else (0.5 if reward == 1 else 0.1)
        old_q = self.q_table.get((prev_state, action), 0.5)
        new_q = old_q + self.alpha * (r + self.gamma * 0.5 - old_q)
        self.q_table[(prev_state, action)] = new_q

agent = QLearningAgent(DIMENSIONS)

# ============================================================================
# 4. WORKFLOW NODES (CONTEXT AWARE)
# ============================================================================

class CaiTIState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    current_dimension: str
    previous_dimension: str
    screened_dimensions: List[str]
    dimension_scores: Dict[str, int]
    phase: Literal["screening", "cbt", "done"]
    cbt_stage: Literal["identify", "challenge", "reframe", "complete"]
    cbt_target_dimension: str
    patient_context: str  # Store the specific patient context here

def node_selector(state: CaiTIState):
    MAX_QUESTIONS = 4 # Keep short for testing
    if len(state["screened_dimensions"]) >= MAX_QUESTIONS:
        return {"phase": "cbt"}

    prev = state.get("current_dimension", "START")
    next_dim = agent.select_next_dimension(prev, state["screened_dimensions"])

    if not next_dim: return {"phase": "cbt"}

    print(f"\n[System] Q-Learning Agent selected: {next_dim}")
    return {
        "current_dimension": next_dim,
        "previous_dimension": prev,
        "screened_dimensions": state["screened_dimensions"] + [next_dim]
    }

def node_questioner(state: CaiTIState):
    """Generates question using SPECIFIC PATIENT CONTEXT."""
    dim = state["current_dimension"]
    context = state["patient_context"] # Loaded from JSON

    prompt = f"""
    You are CaiTI.

    PATIENT PROFILE:
    {context}

    TASK:
    Ask one open-ended question to screen for: '{dim}'.

    RULES:
    - Use the profile to make it personal (e.g. if profile mentions insomnia, ask about sleep specifically).
    - Respect boundaries listed in profile.
    - Keep it under 20 words.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"messages": [response]}

def node_analyzer(state: CaiTIState):
    user_msg = state["messages"][-1].content
    dim = state["current_dimension"]

    prompt = f"""
    Analyze risk for '{dim}': "{user_msg}"
    Output ONLY one number:
    0 = Healthy/Good
    1 = Minor Issue
    2 = High Risk/Concern
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    try:
        score = int(response.content.strip())
    except:
        score = 1

    # Update RL Agent
    prev = state.get("previous_dimension", "START")
    agent.update_q_value(prev, dim, score)
    print(f"   >>> [Analyzer] Score: {score} (Q-Table Updated)")

    scores = state.get("dimension_scores", {})
    scores[dim] = score
    return {"dimension_scores": scores}

def node_mi_reflection(state: CaiTIState):
    dim = state["current_dimension"]
    user_msg = state["messages"][-1].content
    context = state["patient_context"]

    prompt = f"""
    Patient Profile: {context}
    User high-risk response on {dim}: "{user_msg}".
    Provide a 'Simple Reflection' (Motivational Interviewing).
    Validate their feeling. Do NOT give advice yet.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"messages": [response]}

def node_cbt_driver(state: CaiTIState):
    context = state["patient_context"]

    # 1. Initialize
    if not state.get("cbt_target_dimension") or state.get("cbt_stage") is None:
        scores = state.get("dimension_scores", {})
        if not scores: return {"phase": "done"}

        target = max(scores, key=scores.get)
        print(f"\n[System] Starting CBT for: {target}")

        prompt = f"CBT Stage 1: Ask user to identify a negative thought about '{target}'. Profile: {context}"
        response = llm.invoke([HumanMessage(content=prompt)])

        return {
            "cbt_target_dimension": target,
            "cbt_stage": "identify",
            "messages": [response]
        }

    # 2. Advance Stages
    stage = state["cbt_stage"]
    user_input = state["messages"][-1].content

    if stage == "identify":
        prompt = f"CBT Stage 2: User thought: '{user_input}'. Ask them to challenge this thought with evidence."
        next_stage = "challenge"
    elif stage == "challenge":
        prompt = f"CBT Stage 3: User challenge: '{user_input}'. Ask them to reframe it into a balanced thought."
        next_stage = "reframe"
    elif stage == "reframe":
        prompt = f"Conclude the session based on user's reframe: '{user_input}'. Brief encouraging remark."
        next_stage = "complete"
    else:
        return {"phase": "done"}

    response = llm.invoke([HumanMessage(content=prompt)])

    phase = "done" if next_stage == "complete" else "cbt"
    return {"messages": [response], "cbt_stage": next_stage, "phase": phase}

# ============================================================================
# 5. GRAPH BUILD
# ============================================================================

workflow = StateGraph(CaiTIState)
workflow.add_node("selector", node_selector)
workflow.add_node("questioner", node_questioner)
workflow.add_node("analyzer", node_analyzer)
workflow.add_node("mi_reflection", node_mi_reflection)
workflow.add_node("cbt_driver", node_cbt_driver)

workflow.add_edge(START, "selector")
workflow.add_conditional_edges("selector", lambda s: "cbt_driver" if s.get("phase") == "cbt" else "questioner")
workflow.add_edge("questioner", "analyzer")
workflow.add_conditional_edges("analyzer", lambda s: "mi_reflection" if s["dimension_scores"].get(s["current_dimension"], 0) == 2 else "selector")
workflow.add_edge("mi_reflection", "selector")
workflow.add_conditional_edges("cbt_driver", lambda s: END if s.get("phase") == "done" else "cbt_driver")

# Compile with interrupts
app = workflow.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["questioner", "cbt_driver", "mi_reflection"]
)

# ============================================================================
# 6. RUNTIME (PROFILE SELECTION)
# ============================================================================

def run_caiti():
    print("--- CaiTI: Unified Model (RL + MI + CBT + Dynamic Profiles) ---")

    # 1. Load Profiles
    profiles = load_patient_profiles()
    if not profiles: return

    # 2. Display Selection Menu
    print("\nSelect a Patient Profile to Simulate:")
    for p in profiles:
        print(f"[{p['id']}] {p['name']} ({p['condition']}) - Risk: {p['risk_level']}")

    choice = input("\nEnter ID (1-4): ").strip()
    selected_profile = next((p for p in profiles if p["id"] == choice), profiles[0])

    # 3. Serialize Profile to String for Context
    context_str = f"""
    Name: {selected_profile['name']}
    Condition: {selected_profile['condition']}
    Context: {selected_profile['context']}
    Goals: {selected_profile['goals']}
    Boundaries: {selected_profile['boundaries']}
    """

    print(f"\n[System] Loaded Profile: {selected_profile['name']}")


    # Trigger visual reference
    # [Image of reinforcement learning flow diagram]

    # 4. Initialize State
    thread_id = {"configurable": {"thread_id": "105"}}
    initial_input = {
        "messages": [], "screened_dimensions": [], "dimension_scores": {},
        "current_dimension": "START", "phase": "screening",
        "patient_context": context_str  # <--- INJECT CONTEXT HERE
    }

    # 5. Start Loop
    print("[System] Initializing...")
    for event in app.stream(initial_input, thread_id):
        for node, data in event.items():
            if "messages" in data:
                print(f"\nCaiTI: {data['messages'][-1].content}")

    while True:
        try:
            user_text = input("\nYou: ")
            if user_text.lower() in ["quit", "exit"]: break

            app.update_state(thread_id, {"messages": [HumanMessage(content=user_text)]})

            session_done = True
            for event in app.stream(None, thread_id):
                session_done = False
                for node, data in event.items():
                    if "messages" in data:
                        print(f"\nCaiTI: {data['messages'][-1].content}")

            if session_done:
                print("\n[Session Complete]")
                break
        except Exception as e:
            print(f"Error: {e}")
            break

if __name__ == "__main__":
    run_caiti()"
    }
  ],
